# Hunt-and-Kill y Búsqueda no Informada

**Curso:** Inteligencia Artificial · **Referencia:** AIMA, capítulo 3  
**Modalidad:** individual · **Duración sugerida:** 3 semanas  
**Producto:** un cuaderno Jupyter con tres implementaciones comparables

## Problema

Se proporciona solamente un generador de laberintos perfectos mediante **Hunt-and-Kill**. A partir del grafo producido, el estudiante deberá diseñar, implementar, verificar y comparar algoritmos de búsqueda no informada para encontrar un camino entre dos celdas.

El proyecto debe realizarse en tres versiones:

1. **Desde cero:** sin bibliotecas de búsqueda ni de grafos.
2. **SimpleAI:** modelando el laberinto como un `SearchProblem`.
3. **AIMA-Python:** modelándolo como una subclase de `Problem`.

> Este cuaderno es deliberadamente incompleto. No contiene implementaciones Python de DFS, BFS, UCS, IDDFS, búsqueda bidireccional ni Lee. El trabajo evaluado consiste en convertir las especificaciones y el pseudocódigo en soluciones propias.

## Resultados de aprendizaje

Al finalizar, el estudiante estará en capacidad de:

- formular un entorno como espacio de estados;
- separar el problema de la estrategia de búsqueda;
- implementar búsqueda en árbol y búsqueda en grafo;
- justificar completitud, optimalidad y complejidad;
- adaptar un mismo problema a dos bibliotecas de IA;
- construir experimentos reproducibles;
- interpretar el algoritmo de Lee como BFS por frentes de onda;
- defender decisiones de diseño y diagnosticar errores sin depender de una solución externa.

## Condiciones y restricciones

- No se permite `networkx` ni funciones externas que ya resuelvan caminos.
- En la versión desde cero no se permite SimpleAI, AIMA-Python ni otra biblioteca de búsqueda.
- Los estados deben ser inmutables y utilizables como claves de diccionario.
- Cada algoritmo debe devolver solución y métricas, no solamente imprimirlas.
- Todas las versiones deben trabajar con el **mismo grafo**, inicio, meta y costos.
- Los resultados deben ser reproducibles mediante semillas.
- No se acepta una imagen como evidencia: deben entregarse estructuras de datos verificables.
- El estudiante debe citar cualquier recurso consultado y conservar una bitácora breve de decisiones.

### Evidencia de autoría y comprensión

La evaluación incluye:

1. historial de al menos seis hitos o commits;
2. una traza manual sobre un laberinto pequeño;
3. defensa oral individual de 5–8 minutos;
4. modificación en vivo de inicio, meta, semilla o costo;
5. explicación de una decisión que produjo un error y cómo se corrigió.

Durante la defensa se podrá solicitar reconstruir una parte pequeña del algoritmo sin consultar el cuaderno.

## 1. Código suministrado: generador Hunt-and-Kill

Este es el único algoritmo completo entregado. El resultado es un diccionario de adyacencia no dirigido:

```text
grafo[celda] = conjunto de celdas conectadas por un corredor
```

Una celda se representa como `(fila, columna)`.

In [11]:
import random


class LaberintoHuntKill:
    """Genera un laberinto perfecto como grafo no dirigido."""

    DIRECCIONES = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self, filas=25, columnas=25, semilla=2026):
        self.filas = filas
        self.columnas = columnas
        self.rng = random.Random(semilla)
        self.grafo = {
            (fila, columna): set()
            for fila in range(filas)
            for columna in range(columnas)
        }

    def vecinos_geometricos(self, celda):
        fila, columna = celda
        for df, dc in self.DIRECCIONES:
            vecino = (fila + df, columna + dc)
            if (0 <= vecino[0] < self.filas and
                    0 <= vecino[1] < self.columnas):
                yield vecino

    def conectar(self, origen, destino):
        self.grafo[origen].add(destino)
        self.grafo[destino].add(origen)

    def generar(self):
        no_visitadas = set(self.grafo)
        actual = self.rng.choice(tuple(no_visitadas))
        no_visitadas.remove(actual)

        while no_visitadas:
            # KILL: avanzar aleatoriamente hacia una celda no visitada.
            libres = [
                vecino
                for vecino in self.vecinos_geometricos(actual)
                if vecino in no_visitadas
            ]

            if libres:
                siguiente = self.rng.choice(libres)
                self.conectar(actual, siguiente)
                no_visitadas.remove(siguiente)
                actual = siguiente
                continue

            # HUNT: localizar una celda libre vecina del árbol construido.
            candidatas = []
            for celda in no_visitadas:
                visitados = [
                    vecino
                    for vecino in self.vecinos_geometricos(celda)
                    if vecino not in no_visitadas
                ]
                if visitados:
                    candidatas.append((celda, visitados))

            actual, visitados = self.rng.choice(candidatas)
            vecino = self.rng.choice(visitados)
            self.conectar(actual, vecino)
            no_visitadas.remove(actual)

        return self.grafo

### Actividad 1 — Auditoría del generador

Antes de resolver el laberinto:

1. Explique las fases Hunt y Kill señalando las instrucciones correspondientes.
2. Demuestre que el grafo generado es conexo.
3. Justifique por qué contiene exactamente $|V|-1$ aristas.
4. Explique por qué esas dos propiedades implican que es un árbol.
5. Diseñe pruebas para comprobar:
   - simetría de las adyacencias;
   - ausencia de conexiones diagonales;
   - validez de coordenadas;
   - conectividad;
   - ausencia de ciclos;
   - reproducibilidad con una semilla.

No basta con ejecutar el generador: las propiedades deben verificarse automáticamente.

**Respuesta y pruebas del estudiante:**

### 1.1 Las fases Hunt y Kill, señaladas sobre las instrucciones

El generador mantiene una partición de las celdas en **dos conjuntos**:
`no_visitadas`, que llamaremos $U$, y su complemento, las ya incorporadas al
laberinto, que llamaremos $T$. La inicialización, antes del bucle, establece
$T=\{\texttt{actual}\}$ y $U = V \setminus T$:

```python
no_visitadas = set(self.grafo)                     # U = V
actual = self.rng.choice(tuple(no_visitadas))
no_visitadas.remove(actual)                        # T = {actual}
```

**Fase KILL — «camina mientras puedas».** Es un paseo aleatorio que solo pisa
celdas nuevas:

```python
libres = [vecino for vecino in self.vecinos_geometricos(actual)
          if vecino in no_visitadas]               # vecinos de actual que estan en U
if libres:
    siguiente = self.rng.choice(libres)
    self.conectar(actual, siguiente)               # arista: actual en T, siguiente en U
    no_visitadas.remove(siguiente)                 # siguiente pasa a T
    actual = siguiente                             # el paseo continua
    continue
```

El paseo «muere» cuando `libres` queda vacía, es decir cuando todos los vecinos
geométricos de `actual` ya pertenecen a $T$ y el paseo está encerrado.

**Fase HUNT — «busca dónde renacer».** Recorre $U$ buscando celdas que toquen el
árbol ya construido, engancha una y reinicia el paseo desde ella:

```python
candidatas = []
for celda in no_visitadas:                         # celda en U
    visitados = [vecino for vecino in self.vecinos_geometricos(celda)
                 if vecino not in no_visitadas]    # vecinos de celda que estan en T
    if visitados:
        candidatas.append((celda, visitados))

actual, visitados = self.rng.choice(candidatas)    # celda de U vecina de T
vecino = self.rng.choice(visitados)                # su ancla en T
self.conectar(actual, vecino)
no_visitadas.remove(actual)                        # actual pasa a T
```

**Observación sobre la fidelidad al algoritmo original.** El Hunt canónico de la
literatura **escanea la rejilla fila por fila** y toma la *primera* celda no
visitada que tenga un vecino visitado. Esta implementación recolecta **todas** las
candidatas y elige **una al azar** (`self.rng.choice(candidatas)`). No es el mismo
algoritmo y la diferencia es observable:

- el escaneo por filas introduce un sesgo direccional que produce laberintos con
  corredores largos y predominantemente horizontales; el azar lo elimina y
  reparte los puntos de reinicio de forma uniforme;
- el costo cambia: cada Hunt de esta versión es $O(|U|)$ porque recorre todo $U$,
  y como puede haber $\Theta(|V|)$ fases Hunt, el generador es $O(|V|^2)$ en el
  peor caso. Con $|V| = 500$ es irrelevante, pero explica por qué en la sección 10
  el tiempo de **generación** se mide separado del tiempo de **búsqueda**.

### 1.2 Demostración de que el grafo generado es conexo

No se demuestra sobre el resultado final, sino con un **invariante de bucle**.
Sea $E$ el conjunto de aristas creadas hasta el momento.

> **Invariante $I$.** Al inicio de cada iteración del `while`, el subgrafo
> $(T, E)$ es conexo y $|E| = |T| - 1$.

**Base.** Antes del bucle $T=\{\texttt{actual}\}$ y $E=\varnothing$. Un vértice
aislado es conexo y $|E| = 0 = |T| - 1$. $I$ se cumple.

**Paso inductivo.** Todo el argumento se apoya en un único hecho verificable por
inspección: en todo el programa `conectar` se invoca en **dos** lugares, y en
ambos **exactamente un extremo está en $T$ y el otro en $U$**.

| Fase | Invocación | Extremo en $T$ | Extremo en $U$ |
|---|---|---|---|
| KILL | `conectar(actual, siguiente)` | `actual` | `siguiente` |
| HUNT | `conectar(actual, vecino)` | `vecino` | `actual` |

Que `siguiente` esté en $U$ lo garantiza el filtro `if vecino in no_visitadas`;
que `vecino` esté en $T$ lo garantiza el filtro `if vecino not in no_visitadas`.
En consecuencia, cada iteración agrega **un vértice nuevo** a $T$ y **una única
arista** que lo une al resto. Colgar un vértice de un grafo conexo mediante una
arista preserva la conexidad, y el conteo pasa de $|T|-1$ a $|T'|-1$ con
$|T'| = |T|+1$. $I$ se preserva.

**Terminación.** Cada iteración ejecuta exactamente un `no_visitadas.remove(...)`
—uno en la rama KILL y uno en la rama HUNT—, de modo que $|U|$ decrece en $1$ por
iteración. El bucle ejecuta exactamente $|V|-1$ iteraciones y termina.

**Conclusión.** Al salir del bucle $U=\varnothing$, luego $T=V$, y por $I$ el
grafo $(V,E)$ es conexo. $\blacksquare$

**Condición oculta de la que depende la corrección.** La fase Hunt asume que
`candidatas` nunca está vacía; si lo estuviera, `rng.choice([])` lanzaría
`IndexError`. Nunca ocurre porque mientras $U\neq\varnothing$ y $T\neq\varnothing$
la **rejilla geométrica** es conexa, y por tanto existe al menos una arista de la
rejilla que cruza el corte $(T, U)$. La corrección del generador **depende de que
la rejilla de partida sea conexa**: si el enunciado admitiera celdas bloqueadas de
antemano, el generador fallaría con `IndexError` en lugar de producir un laberinto
parcial.

### 1.3 Justificación de que contiene exactamente $|V|-1$ aristas

Del apartado anterior: el bucle ejecuta $|V|-1$ iteraciones y cada una invoca
`conectar` exactamente una vez, luego hay $|V|-1$ invocaciones.

Falta cerrar un hueco que suele pasarse por alto: `conectar` usa `set.add`, que
**descarta duplicados en silencio**. Si algún par se conectara dos veces,
tendríamos $|V|-1$ invocaciones pero **menos** de $|V|-1$ aristas. No ocurre
porque toda arista creada es incidente al vértice que entra a $T$ en esa
iteración, y cada vértice entra a $T$ una sola vez; las $|V|-1$ aristas son por
tanto distintas dos a dos. $\blacksquare$

Formulación equivalente y directamente verificable en código, ya que el
diccionario almacena cada arista en ambos sentidos:

$$\sum_{v \in V} \bigl|\,\texttt{grafo}[v]\,\bigr| = 2\,(|V|-1)$$

### 1.4 Por qué esas dos propiedades implican que es un árbol

Teorema estándar de teoría de grafos: para un grafo de $n$ vértices, de las tres
propiedades

1. es conexo,
2. es acíclico,
3. tiene exactamente $n-1$ aristas,

**cualesquiera dos implican la tercera**. Habiendo demostrado (1) en 1.2 y (3) en
1.3, se concluye (2), y un grafo conexo y acíclico es por definición un **árbol**.

La intuición del porqué: un grafo conexo sobre $n$ vértices necesita **al menos**
$n-1$ aristas; si tiene exactamente $n-1$ no le sobra ninguna, y un ciclo sería
precisamente una arista sobrante respecto de un árbol de expansión.

**Consecuencia que gobierna el resto del taller.** En un árbol existe **un único
camino simple** entre dos vértices cualesquiera. De ahí se sigue que:

- el laberinto es *perfecto*: sin ciclos y sin regiones inalcanzables;
- **DFS, BFS y UCS devolverán el mismo camino**, no porque los algoritmos sean
  equivalentes, sino porque no existe ninguna alternativa que escoger;
- por tanto un laberinto perfecto **oculta las diferencias de calidad de ruta**, y
  la sección 9 (ciclos y costos) no es un añadido opcional: sin ella la
  optimalidad de UCS es inobservable y ningún experimento puede distinguirla de
  BFS.

### 1.5 Diseño de las pruebas

El enunciado exige verificar seis propiedades. La batería implementa **once**: las
seis exigidas y cinco adicionales que atrapan errores que las primeras no ven.

| # | Propiedad | Qué error atraparía | Exigida |
|---|---|---|---|
| 1 | Simetría: $v \in G[u] \iff u \in G[v]$ | un `conectar` escrito como arista dirigida | sí |
| 2 | Ortogonalidad: $\lvert\Delta f\rvert + \lvert\Delta c\rvert = 1$ | vecindad geométrica con diagonales | sí |
| 3 | Dominio: claves $=$ rejilla, sin coordenadas fuera de rango | celdas faltantes, sobrantes o inválidas | sí |
| 4 | Conectividad por inundación propia | regiones aisladas | sí |
| 5 | Aciclicidad por DFS con arista de retorno | ciclos, **detectados directamente** | sí |
| 6 | Reproducibilidad: misma semilla $\Rightarrow$ mismo grafo | azar no sembrado | sí |
| 7 | $\lvert E\rvert = \lvert V\rvert-1$, $\sum \deg = 2\lvert E\rvert$, $\deg \le 4$ | verifica el teorema de 1.4 | no |
| 8 | Unicidad del camino simple | que sea árbol *en la práctica*, no solo en la demostración | no |
| 9 | Ausencia de lazos ($u \notin G[u]$) | caso degenerado | no |
| 10 | Reproducibilidad **entre procesos** con `PYTHONHASHSEED` distinto | dependencia del hash del proceso | no |
| 11 | `generar()` es de un solo uso | contrato implícito de la clase | no |

Tres decisiones de diseño merecen justificación explícita.

**(a) Las utilidades de verificación son independientes de los buscadores.** La
inundación y el detector de ciclos de esta sección se escriben aparte y no se
reutilizan de la Versión 1. Si validáramos el grafo con el mismo buscador que
luego va a correr sobre ese grafo, un error compartido por ambos **se cancelaría**
y la prueba pasaría estando las dos cosas mal. Una prueba solo tiene valor si
puede fallar de forma independiente de aquello que verifica.

**(b) La aciclicidad se comprueba dos veces por caminos distintos.** La
comprobación 7 la deduce del teorema ($|E| = |V|-1$ junto con conexidad) y la 5 la
detecta directamente buscando una arista de retorno en una DFS. Duplicar el
argumento evita que toda la conclusión «es un árbol» dependa de una sola línea de
código.

**(c) Dos propiedades no son universales, y reconocerlo es parte de la
auditoría.** Se marcan como **no aplicables** en lugar de forzarlas o
silenciarlas:

- La **sensibilidad a la semilla** (comprobación 6, parte «semilla distinta
  $\Rightarrow$ grafo distinto») no puede exigirse en una rejilla $1\times N$ o
  $N\times 1$: la rejilla *es* un camino y su único árbol de expansión es ella
  misma, de modo que ninguna semilla puede producir un laberinto diferente.
- La **no idempotencia** (comprobación 11) tampoco se observa en esas rejillas,
  por la razón que se explica en 1.6.

**Por qué existe la comprobación 10.** El generador ejecuta
`self.rng.choice(tuple(no_visitadas))` y en la fase Hunt recorre `no_visitadas`
con un `for`: es decir, **el orden de iteración de un conjunto alimenta al
generador aleatorio**. Eso resulta reproducible únicamente porque las celdas son
tuplas de enteros, y CPython no aleatoriza el hash de enteros ni de tuplas de
enteros. Si las celdas se representaran como cadenas —por ejemplo `"3,4"`—, la
variable de entorno `PYTHONHASHSEED` cambiaría el orden del conjunto en cada
proceso y **la semilla dejaría de garantizar nada**. La comprobación 6 corre en un
solo proceso y es estructuralmente incapaz de detectarlo; la 10 lanza tres
subprocesos con `PYTHONHASHSEED` $\in \{0, 1, 42\}$ y compara huellas SHA-256.
Esta es la razón de fondo por la que el enunciado insiste en que *«los estados
deben ser inmutables y utilizables como claves de diccionario»*: la elección de
representación no es cosmética, condiciona la reproducibilidad.

**(d) La comprobación 10 incluye su propio contra-experimento.** Una prueba que
no puede fallar no vale nada. Si `PYTHONHASHSEED` no llegara al subproceso —por
un error al construir el entorno, por ejemplo—, las tres huellas coincidirían por
una razón trivial y la comprobación pasaría **sin haber comprobado nada**. Para
descartarlo se ejecuta en paralelo el mismo experimento con las celdas
representadas como **cadenas**, donde el orden del conjunto *sí* debe variar. La
comprobación exige las dos condiciones simultáneamente: invariancia con tuplas de
enteros y variación con cadenas. Medido: las tres huellas con tuplas coinciden y
las tres con cadenas son distintas entre sí.

### 1.6 Hallazgos de la auditoría

**Hallazgo 1 — `generar()` no es idempotente; el objeto es de un solo uso.**
Una segunda llamada a `generar()` sobre el mismo objeto reinicia `no_visitadas` a
partir de `self.grafo`, pero **las aristas de la primera pasada siguen allí**. La
segunda pasada agrega otras $|V|-1$ aristas y el resultado deja de ser un árbol.
Medido sobre la instancia individual: la primera pasada deja $499$ aristas
$=|V|-1$; la segunda deja $722$ y aparece un ciclo. No es un error del código
suministrado sino un **contrato implícito** de la clase, y por eso todo el
cuaderno construye una instancia nueva por laberinto y nunca reutiliza una.

**Hallazgo 2 — el defecto anterior no es universal, y lo descubrió la propia
prueba.** La comprobación 11 falló al ejecutarla sobre una rejilla $1\times 5$.
El diagnóstico: en una rejilla degenerada la única arista posible entre dos celdas
vecinas ya existe tras la primera pasada, así que la segunda intenta recrear
**las mismas** aristas y `set.add` las descarta en silencio; el conteo permanece
en $|V|-1$ y no aparece ningún ciclo. La no idempotencia se manifiesta únicamente
si la rejilla posee ciclos propios, es decir si $\text{filas} \ge 2$ y
$\text{columnas} \ge 2$. La prueba estaba mal planteada, no el generador: se
corrigió marcándola no aplicable en rejillas degeneradas. Queda registrado en
`BITACORA.md` como **D-09**.

**Hallazgo 3 — `inspect.getsource` no sirve para leer una celda bajo
`nbconvert`.** La comprobación 10 necesita el código fuente de
`LaberintoHuntKill` para re-ejecutarlo en un subproceso limpio. La primera
implementación usaba `inspect.getsource(clase)`, que funciona en una sesión
interactiva de Jupyter porque IPython registra el código de cada celda en
`linecache`. Al comprobar que el cuaderno corre de arriba abajo con
`jupyter nbconvert --execute` —justamente el modo que exige el enunciado—
`inspect.getsource` lanzó `OSError: source code not available` y la comprobación
quedó marcada como no aplicable: la auditoría *parecía* pasar con un agujero
dentro. Corrección: una función `fuente_del_generador` que intenta
`inspect.getsource` y, si falla, **lee el propio `.ipynb`** del directorio de
trabajo y localiza la celda que define la clase **por el nombre de la clase, no
por su índice**, de modo que reordenar celdas no rompa la prueba. Registrado como
**D-10**. La lección general es que una comprobación que se degrada a «no
aplicable» en silencio es peor que una que falla: por eso el resumen de la
auditoría imprime explícitamente cuántas comprobaciones quedaron sin aplicar.

**Hallazgo 4 — la distribución de grados describe la textura del laberinto.** En
la instancia individual: $58$ celdas de grado $1$ (callejones sin salida), $387$
de grado $2$ (corredor recto o curva), $54$ de grado $3$ (bifurcación) y $1$ de
grado $4$ (cruce). Un grado máximo de $4$ es consistente con la vecindad
ortogonal, y la abundancia de grado $2$ con el paseo aleatorio de la fase Kill,
que produce corredores largos. Estas cifras se retoman en la sección 11 para
explicar el factor de ramificación efectivo $b$: aunque $b \le 4$ en teoría, el
$b$ **medio observado** es cercano a $2$, y esa es la razón por la que las cotas
$O(b^d)$ sobreestiman ampliamente el número real de nodos expandidos.

In [ ]:
# ===========================================================================
# Actividad 1 - Auditoria automatica del generador Hunt-and-Kill
#
# Las propiedades de un laberinto no se "ven" en un dibujo: se comprueban.
# Cada comprobacion devuelve un objeto `Comprobacion` con evidencia
# verificable -estructuras de datos, no imagenes-, como exige el enunciado.
#
# DECISION DE DISENO (D-07): la inundacion y el detector de ciclos de esta
# seccion son utilidades de VERIFICACION, escritas aparte y a proposito
# independientes de los buscadores de la Version 1. Si validaramos el grafo
# con el mismo buscador que luego corre sobre ese grafo, un error compartido
# se cancelaria y la prueba pasaria estando ambos mal.
# ===========================================================================

from collections import deque
from dataclasses import dataclass, field
import glob
import hashlib
import inspect
import io
import json
import os
import subprocess
import sys
import tempfile


@dataclass
class Comprobacion:
    """Resultado de una sola propiedad auditada."""
    nombre: str
    aprobada: bool
    detalle: str
    evidencia: dict = field(default_factory=dict)
    aplicable: bool = True   # una propiedad puede no tener sentido en un caso limite


# --------------------------------------------------------------------------
# Utilidades sobre el grafo de adyacencia
# --------------------------------------------------------------------------

def aristas_canonicas(grafo):
    """Conjunto de aristas no dirigidas en forma canonica (menor, mayor).

    El grafo guarda cada arista dos veces (u en G[v] y v en G[u]). Ordenar el
    par elimina esa duplicidad y permite contar aristas de verdad.
    """
    return {(u, v) if u <= v else (v, u)
            for u, vecinos in grafo.items()
            for v in vecinos}


def grados(grafo):
    """Grado de cada celda: cuantos corredores salen de ella."""
    return {celda: len(vecinos) for celda, vecinos in grafo.items()}


def huella(grafo):
    """SHA-256 de una serializacion canonica del grafo.

    Sirve para comparar dos grafos por igualdad exacta sin volcar 500 celdas.
    La serializacion se ordena, asi que no depende del orden de iteracion de
    los diccionarios ni de los conjuntos.
    """
    texto = ";".join(
        "{0}->{1}".format(celda, sorted(grafo[celda]))
        for celda in sorted(grafo)
    )
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def inundar(grafo, origen):
    """Inundacion propia (BFS sin metricas) para medir alcanzabilidad.

    No es el BFS de la Version 1: aqui solo interesa el CONJUNTO alcanzado,
    no el camino ni el contrato de metricas. Ver D-07.
    """
    vistos = {origen}
    cola = deque([origen])
    while cola:
        actual = cola.popleft()
        for vecino in grafo[actual]:
            if vecino not in vistos:
                vistos.add(vecino)
                cola.append(vecino)
    return vistos


def buscar_ciclo(grafo):
    """Devuelve una arista de retorno si existe un ciclo, o None.

    DFS iterativa marcando al APILAR y saltando la arista hacia el padre.
    En un grafo no dirigido y simple, encontrar un vecino ya visitado que no
    sea el padre significa que esa arista cierra un ciclo con el arbol DFS.
    """
    visitados = set()
    for raiz in grafo:
        if raiz in visitados:
            continue
        visitados.add(raiz)
        pila = [(raiz, None)]
        while pila:
            nodo, padre = pila.pop()
            for vecino in grafo[nodo]:
                if vecino == padre:
                    continue
                if vecino in visitados:
                    return (nodo, vecino)
                visitados.add(vecino)
                pila.append((vecino, nodo))
    return None


def caminos_simples(grafo, inicio, meta, tope=2):
    """Enumera hasta `tope` caminos simples entre dos celdas.

    Se corta en `tope` porque solo interesa distinguir "exactamente uno" de
    "mas de uno": enumerar todos los caminos de un grafo con ciclos es
    exponencial y aqui no hace falta.
    """
    encontrados = []
    pila = [(inicio, (inicio,))]
    while pila and len(encontrados) < tope:
        nodo, camino = pila.pop()
        if nodo == meta:
            encontrados.append(camino)
            continue
        for vecino in sorted(grafo[nodo]):
            if vecino not in camino:          # camino SIMPLE: sin repetir
                pila.append((vecino, camino + (vecino,)))
    return encontrados

In [ ]:
# --------------------------------------------------------------------------
# Las once comprobaciones
#
# Dos de ellas NO son universales, y reconocerlo es parte de la auditoria:
#   - la sensibilidad a la semilla no puede exigirse en una rejilla 1xN,
#     porque un camino tiene un unico arbol de expansion posible;
#   - la no idempotencia de `generar()` no puede observarse con |V| = 1,
#     porque no hay ninguna arista que duplicar.
# Se marcan como NO APLICABLES en lugar de forzarlas o de silenciarlas.
# --------------------------------------------------------------------------

def comprobar_simetria(grafo):
    """1. v en G[u] <=> u en G[v]. Atrapa un `conectar` escrito como dirigido."""
    fallos = [(u, v) for u, vecinos in grafo.items()
              for v in vecinos if u not in grafo.get(v, ())]
    return Comprobacion(
        "1. Simetria de las adyacencias",
        not fallos,
        "las {0} aristas son bidireccionales".format(len(aristas_canonicas(grafo)))
        if not fallos else "{0} adyacencias asimetricas".format(len(fallos)),
        {"pares_asimetricos": fallos[:5]},
    )


def comprobar_ortogonalidad(grafo):
    """2. Sin diagonales: la distancia Manhattan de toda arista debe ser 1."""
    fallos = [(u, v) for u, vecinos in grafo.items() for v in vecinos
              if abs(u[0] - v[0]) + abs(u[1] - v[1]) != 1]
    return Comprobacion(
        "2. Ausencia de conexiones diagonales",
        not fallos,
        "toda arista tiene distancia Manhattan 1"
        if not fallos else "{0} aristas no ortogonales".format(len(fallos)),
        {"aristas_invalidas": fallos[:5]},
    )


def comprobar_dominio(grafo, filas, columnas):
    """3. El grafo cubre exactamente la rejilla filas x columnas, sin extras."""
    esperadas = {(f, c) for f in range(filas) for c in range(columnas)}
    presentes = set(grafo)
    fuera = {v for vecinos in grafo.values() for v in vecinos} - esperadas
    ok = presentes == esperadas and not fuera
    return Comprobacion(
        "3. Validez de coordenadas y cobertura",
        ok,
        "{0} celdas = {1}x{2}, ninguna coordenada fuera de rango".format(
            len(presentes), filas, columnas)
        if ok else "faltan {0}, sobran {1}, vecinos invalidos {2}".format(
            len(esperadas - presentes), len(presentes - esperadas), len(fuera)),
        {"faltantes": sorted(esperadas - presentes)[:5],
         "sobrantes": sorted(presentes - esperadas)[:5],
         "vecinos_fuera_de_rango": sorted(fuera)[:5]},
    )


def comprobar_conectividad(grafo):
    """4. Una sola componente: desde cualquier celda se alcanzan todas."""
    origen = next(iter(grafo))
    alcanzadas = inundar(grafo, origen)
    ok = len(alcanzadas) == len(grafo)
    return Comprobacion(
        "4. Conectividad (una sola componente)",
        ok,
        "desde {0} se alcanzan las {1} celdas".format(origen, len(alcanzadas))
        if ok else "solo {0} de {1} celdas alcanzables".format(
            len(alcanzadas), len(grafo)),
        {"origen": origen, "alcanzadas": len(alcanzadas), "total": len(grafo),
         "aisladas_ejemplo": sorted(set(grafo) - alcanzadas)[:5]},
    )


def comprobar_aciclicidad(grafo):
    """5. Sin ciclos, detectado DIRECTAMENTE y no por el conteo de aristas.

    Importa que sea directo: el teorema "conexo + |V|-1 aristas => arbol" ya
    se usa en la comprobacion 7. Detectar el ciclo con una DFS independiente
    evita que toda la conclusion dependa de un unico argumento.
    """
    arista = buscar_ciclo(grafo)
    return Comprobacion(
        "5. Ausencia de ciclos (DFS con arista de retorno)",
        arista is None,
        "ninguna arista de retorno en la DFS"
        if arista is None else "ciclo cerrado por la arista {0}".format(arista),
        {"arista_de_retorno": arista},
    )


def comprobar_reproducibilidad(clase, filas, columnas, semilla,
                               exigir_sensibilidad=True):
    """6. Misma semilla -> grafo identico; semilla distinta -> grafo distinto.

    `exigir_sensibilidad=False` en rejillas degeneradas (1xN o Nx1): un camino
    tiene un unico arbol de expansion, asi que ninguna semilla puede producir
    un laberinto distinto. Exigirlo alli seria una prueba mal planteada, no un
    defecto del generador.
    """
    h1 = huella(clase(filas, columnas, semilla).generar())
    h2 = huella(clase(filas, columnas, semilla).generar())
    h3 = huella(clase(filas, columnas, semilla + 1).generar())

    determinista = h1 == h2
    sensible = h1 != h3
    ok = determinista and (sensible or not exigir_sensibilidad)
    if not determinista:
        detalle = "la misma semilla produjo dos grafos distintos"
    elif exigir_sensibilidad and not sensible:
        detalle = "semillas distintas produjeron el mismo grafo"
    elif exigir_sensibilidad:
        detalle = "semilla {0} reproduce la huella; {1} produce otra".format(
            semilla, semilla + 1)
    else:
        detalle = ("determinista; sensibilidad no exigida (rejilla degenerada: "
                   "el arbol de expansion es unico)")
    return Comprobacion(
        "6. Reproducibilidad con semilla", ok, detalle,
        {"huella_semilla": h1[:16], "huella_repeticion": h2[:16],
         "huella_semilla_mas_uno": h3[:16],
         "determinista": determinista, "sensible_a_la_semilla": sensible},
    )


def comprobar_conteo_aristas(grafo):
    """7. |E| == |V|-1, suma de grados == 2|E|, grado maximo <= 4."""
    aristas = aristas_canonicas(grafo)
    gs = grados(grafo)
    suma_grados = sum(gs.values())
    n = len(grafo)
    grado_max = max(gs.values())
    ok = (len(aristas) == n - 1 and suma_grados == 2 * (n - 1) and grado_max <= 4)
    return Comprobacion(
        "7. Conteo de aristas |E| = |V|-1 y grados <= 4", ok,
        "|V|={0}, |E|={1}, suma de grados={2}, grado max={3}".format(
            n, len(aristas), suma_grados, grado_max),
        {"vertices": n, "aristas": len(aristas), "esperadas": n - 1,
         "suma_grados": suma_grados, "grado_maximo": grado_max,
         "distribucion_grados": {g: list(gs.values()).count(g)
                                 for g in sorted(set(gs.values()))}},
    )


def comprobar_unicidad_camino(grafo, inicio, meta):
    """8. En un arbol existe EXACTAMENTE un camino simple entre dos celdas.

    Esta es la propiedad que hace "perfecto" al laberinto, y la razon por la
    que DFS, BFS y UCS devolveran el MISMO camino: no hay alternativa que
    escoger. Sin la seccion 9 (ciclos) la optimalidad seria invisible.
    """
    caminos = caminos_simples(grafo, inicio, meta, tope=2)
    ok = len(caminos) == 1
    return Comprobacion(
        "8. Unicidad del camino simple", ok,
        "un unico camino de {0} a {1}, de {2} celdas".format(
            inicio, meta, len(caminos[0])) if ok
        else "se encontraron {0} caminos entre {1} y {2}".format(
            len(caminos), inicio, meta),
        {"numero_de_caminos": len(caminos),
         "longitud_en_celdas": len(caminos[0]) if caminos else 0,
         "primeras_celdas": list(caminos[0][:5]) if caminos else []},
    )


def comprobar_sin_lazos(grafo):
    """9. Ninguna celda es vecina de si misma."""
    lazos = [u for u, vecinos in grafo.items() if u in vecinos]
    return Comprobacion(
        "9. Ausencia de lazos", not lazos,
        "ninguna celda se conecta consigo misma"
        if not lazos else "{0} lazos".format(len(lazos)),
        {"lazos": lazos[:5]},
    )


def fuente_del_generador(clase):
    """Devuelve el codigo fuente de la clase generadora, o None.

    Hace falta para re-ejecutar el generador en un SUBPROCESO limpio, y
    obtenerlo no es trivial cuando la clase se define en una celda:

    1. `inspect.getsource` funciona en una sesion interactiva de IPython
       -que registra el codigo de cada celda en `linecache`- pero FALLA con
       OSError bajo `jupyter nbconvert --execute`, que es precisamente el modo
       en que se comprueba que el cuaderno corre de arriba abajo.
    2. Como respaldo se lee el propio `.ipynb` del directorio de trabajo y se
       localiza la celda que define la clase por su nombre, no por su indice,
       para que reordenar celdas no rompa la prueba.
    """
    try:
        return inspect.getsource(clase)
    except (OSError, TypeError):
        pass

    firma = "class " + clase.__name__
    for ruta in sorted(glob.glob("*.ipynb")):
        try:
            cuaderno = json.load(io.open(ruta, encoding="utf-8"))
        except (ValueError, OSError):
            continue
        for celda in cuaderno.get("cells", []):
            fuente = "".join(celda.get("source", []))
            if firma in fuente:
                return fuente
    return None


def _ejecutar_en_subprocesos(guion, valores=("0", "1", "42")):
    """Corre un guion una vez por cada valor de PYTHONHASHSEED."""
    ruta = os.path.join(tempfile.mkdtemp(), "replica.py")
    with io.open(ruta, "w", encoding="utf-8") as archivo:
        archivo.write(guion)
    salidas = {}
    for valor in valores:
        entorno = dict(os.environ, PYTHONHASHSEED=valor)
        proceso = subprocess.run([sys.executable, ruta], capture_output=True,
                                 text=True, env=entorno, check=True)
        salidas[valor] = proceso.stdout.strip()
    return salidas


def comprobar_reproducibilidad_entre_procesos(clase, filas, columnas, semilla):
    """10. Misma semilla en procesos distintos con PYTHONHASHSEED distinto.

    POR QUE ESTA PRUEBA EXISTE (D-08):
    el generador ejecuta `rng.choice(tuple(no_visitadas))` y en la fase Hunt
    recorre `no_visitadas` con un `for`. Es decir, EL ORDEN DE ITERACION DE UN
    CONJUNTO ALIMENTA AL GENERADOR ALEATORIO. Eso resulta reproducible solo
    porque las celdas son tuplas de enteros, y CPython no aleatoriza el hash
    de enteros ni de tuplas de enteros. Si las celdas fueran cadenas -por
    ejemplo "3,4"-, `PYTHONHASHSEED` cambiaria el orden del conjunto en cada
    proceso y la semilla NO garantizaria nada. La comprobacion 6 corre en un
    solo proceso y es estructuralmente incapaz de detectarlo.

    LA PRUEBA INCLUYE SU PROPIO CONTRA-EXPERIMENTO (D-10). Una prueba que no
    puede fallar no vale nada: si `PYTHONHASHSEED` no llegara al subproceso,
    las tres huellas coincidirian por una razon trivial y la prueba pasaria
    sin haber comprobado nada. Por eso se corre en paralelo el MISMO
    experimento con celdas representadas como cadenas, donde el orden del
    conjunto SI debe cambiar. La comprobacion exige las dos cosas a la vez:
    invariancia con tuplas y variacion con cadenas.
    """
    fuente = fuente_del_generador(clase)
    if fuente is None:
        return Comprobacion(
            "10. Reproducibilidad entre procesos (PYTHONHASHSEED)", True,
            "no aplicable: no se pudo recuperar el fuente de la clase",
            {"motivo": "ni inspect.getsource ni la lectura del .ipynb"},
            aplicable=False)

    guion_generador = (
        "import random, hashlib\n" + fuente + "\n"
        + "g = {0}({1}, {2}, {3}).generar()\n".format(
            clase.__name__, filas, columnas, semilla)
        + 'texto = ";".join("{0}->{1}".format(c, sorted(g[c])) for c in sorted(g))\n'
        + 'print(hashlib.sha256(texto.encode("utf-8")).hexdigest())\n'
    )

    # Contra-experimento: las mismas celdas, pero como CADENAS.
    guion_cadenas = (
        "import hashlib\n"
        + "celdas = ['{0},{1}'.format(f, c)\n"
        + "          for f in range({0}) for c in range({1})]\n".format(
            filas, columnas)
        + "orden = ';'.join(set(celdas))\n"
        + 'print(hashlib.sha256(orden.encode("utf-8")).hexdigest())\n'
    )

    con_tuplas = _ejecutar_en_subprocesos(guion_generador)
    con_cadenas = _ejecutar_en_subprocesos(guion_cadenas)
    referencia = huella(clase(filas, columnas, semilla).generar())

    invariante = len(set(con_tuplas.values())) == 1
    coincide = con_tuplas["0"] == referencia
    contraste = len(set(con_cadenas.values())) > 1
    ok = invariante and coincide and contraste

    if not invariante:
        detalle = "el laberinto depende del PYTHONHASHSEED del proceso"
    elif not coincide:
        detalle = "el subproceso no reproduce el grafo de este proceso"
    elif not contraste:
        detalle = ("PRUEBA VACIA: con cadenas el orden tampoco cambio, luego "
                   "PYTHONHASHSEED no esta surtiendo efecto")
    else:
        detalle = ("invariante con tuplas de enteros y variable con cadenas: "
                   "{0} ordenes distintos".format(len(set(con_cadenas.values()))))

    return Comprobacion(
        "10. Reproducibilidad entre procesos (PYTHONHASHSEED)", ok, detalle,
        {"huellas_con_tuplas": {k: v[:16] for k, v in con_tuplas.items()},
         "huella_en_este_proceso": referencia[:16],
         "huellas_del_contraexperimento_con_cadenas":
             {k: v[:16] for k, v in con_cadenas.items()},
         "invariante_con_tuplas": invariante,
         "varia_con_cadenas": contraste},
    )


def comprobar_un_solo_uso(clase, filas, columnas, semilla):
    """11. Fragilidad documentada: `generar()` NO es idempotente.

    Llamar `generar()` dos veces sobre el MISMO objeto reinicia
    `no_visitadas` desde `self.grafo`, pero las aristas de la primera pasada
    siguen ahi. La segunda pasada agrega otras |V|-1 aristas y CREA CICLOS:
    el resultado deja de ser un arbol.

    La prueba no denuncia un error del codigo suministrado: documenta un
    contrato implicito de la clase. Todo el cuaderno construye una instancia
    nueva por laberinto y nunca reutiliza una.

    PERO EL DEFECTO NO ES UNIVERSAL (hallazgo, ver D-09). En una rejilla
    degenerada 1xN o Nx1 la segunda pasada NO rompe nada: la rejilla es un
    camino, su unico arbol de expansion es ella misma, asi que las aristas que
    la segunda pasada intenta crear YA EXISTEN y `set.add` las descarta en
    silencio. El conteo sigue en |V|-1 y no aparece ningun ciclo.
    Conclusion: la no idempotencia se observa solo si la rejilla tiene ciclos
    propios, es decir si filas >= 2 y columnas >= 2.
    """
    n = filas * columnas
    if filas < 2 or columnas < 2:
        return Comprobacion(
            "11. `generar()` es de un solo uso (fragilidad documentada)", True,
            "no aplicable: en una rejilla {0}x{1} el arbol de expansion es "
            "unico, asi que repetir `generar()` reescribe las mismas aristas "
            "y `set.add` las descarta".format(filas, columnas),
            {"vertices": n, "filas": filas, "columnas": columnas},
            aplicable=False)

    laberinto = clase(filas, columnas, semilla)
    primera = huella(laberinto.generar())
    aristas_primera = len(aristas_canonicas(laberinto.grafo))
    laberinto.generar()                   # segunda llamada: rompe el arbol
    aristas_segunda = len(aristas_canonicas(laberinto.grafo))
    ciclo = buscar_ciclo(laberinto.grafo)

    ok = (aristas_primera == n - 1 and aristas_segunda > n - 1
          and ciclo is not None)
    return Comprobacion(
        "11. `generar()` es de un solo uso (fragilidad documentada)", ok,
        "1a pasada {0} aristas (=|V|-1); 2a pasada {1} y aparece el ciclo "
        "{2}".format(aristas_primera, aristas_segunda, ciclo) if ok
        else "el comportamiento observado no coincide con el esperado",
        {"aristas_primera_pasada": aristas_primera,
         "aristas_segunda_pasada": aristas_segunda,
         "esperado_arbol": n - 1, "ciclo_detectado": ciclo,
         "huella_primera_pasada": primera[:16]},
    )

In [ ]:
# --------------------------------------------------------------------------
# Ejecutor de la bateria
# --------------------------------------------------------------------------

def auditar(clase, filas, columnas, semilla, inicio=None, meta=None,
            grafo=None, entre_procesos=True):
    """Corre la bateria y DEVUELVE la lista de resultados; no imprime.

    Quien llama decide como presentarlos: el enunciado exige evidencia
    verificable, no salida por pantalla.

    - `inicio`/`meta`: por omision, esquinas opuestas de la rejilla.
    - `grafo`: permite auditar un grafo ya generado en lugar de generar otro.
    - `entre_procesos`: la comprobacion 10 lanza tres subprocesos, asi que se
      desactiva en las configuraciones secundarias por costo, no por dudas.
    """
    if grafo is None:
        grafo = clase(filas, columnas, semilla).generar()
    if inicio is None:
        inicio = (0, 0)
    if meta is None:
        meta = (filas - 1, columnas - 1)

    degenerada = filas < 2 or columnas < 2

    resultados = [
        comprobar_simetria(grafo),
        comprobar_ortogonalidad(grafo),
        comprobar_dominio(grafo, filas, columnas),
        comprobar_conectividad(grafo),
        comprobar_aciclicidad(grafo),
        comprobar_reproducibilidad(clase, filas, columnas, semilla,
                                   exigir_sensibilidad=not degenerada),
        comprobar_conteo_aristas(grafo),
        comprobar_unicidad_camino(grafo, inicio, meta),
        comprobar_sin_lazos(grafo),
    ]
    if entre_procesos:
        resultados.append(
            comprobar_reproducibilidad_entre_procesos(
                clase, filas, columnas, semilla))
    resultados.append(comprobar_un_solo_uso(clase, filas, columnas, semilla))
    return resultados


def imprimir_auditoria(resultados, titulo="AUDITORIA DEL GENERADOR HUNT-AND-KILL"):
    """Tabla legible + `assert`: si algo falla, el cuaderno se detiene aqui."""
    ancho = max(len(r.nombre) for r in resultados)
    print(titulo)
    print("-" * (ancho + 62))
    for r in resultados:
        if not r.aplicable:
            marca = "N/A "
        elif r.aprobada:
            marca = " OK "
        else:
            marca = "FALL"
        print("[{0}] {1}  {2}".format(marca, r.nombre.ljust(ancho), r.detalle))
    print("-" * (ancho + 62))

    aplicables = [r for r in resultados if r.aplicable]
    fallidas = [r.nombre for r in aplicables if not r.aprobada]
    print("{0}/{1} comprobaciones aplicables aprobadas ({2} no aplicables)".format(
        len(aplicables) - len(fallidas), len(aplicables),
        len(resultados) - len(aplicables)))
    assert not fallidas, "comprobaciones fallidas: {0}".format(fallidas)
    return resultados

In [ ]:
# --------------------------------------------------------------------------
# Ejecucion de la bateria
#
# Se audita el generador en CINCO configuraciones y no solo en la instancia
# individual: un generador correcto debe seguir siendolo en los casos limite,
# y ahi es donde aparecieron los dos hallazgos del apartado 1.6.
#
# La instancia individual se vuelve a auditar en la seccion 2, una vez
# definidos formalmente sus parametros, para no crear aqui una dependencia
# hacia adelante (el cuaderno debe ejecutarse de arriba abajo).
# --------------------------------------------------------------------------

CONFIGURACIONES_AUDITORIA = [
    (1,  1,  97950, "caso limite: una sola celda"),
    (1,  5,  97950, "caso limite: rejilla degenerada 1xN"),
    (5,  1,  97950, "caso limite: rejilla degenerada Nx1"),
    (3,  3,  97950, "laberinto minimo no trivial"),
    (20, 25, 97950, "tamano de la instancia individual"),
]

AUDITORIAS = {}
for _filas, _columnas, _semilla, _nota in CONFIGURACIONES_AUDITORIA:
    # La comprobacion 10 lanza tres subprocesos; se activa solo en la
    # configuracion principal por costo, no por dudas sobre su validez.
    _principal = (_filas, _columnas) == (20, 25)
    AUDITORIAS[(_filas, _columnas)] = imprimir_auditoria(
        auditar(LaberintoHuntKill, _filas, _columnas, _semilla,
                entre_procesos=_principal),
        "AUDITORIA {0}x{1}  -  {2}".format(_filas, _columnas, _nota))
    print()

# El enunciado exige evidencia verificable, no solo salida por pantalla:
# `AUDITORIAS` conserva los objetos Comprobacion con su campo `evidencia`.
print("Evidencia estructurada de tres comprobaciones sobre la rejilla 20x25:")
for _comprobacion in AUDITORIAS[(20, 25)]:
    if _comprobacion.nombre[:2] in ("7.", "10", "11"):
        print(" ", _comprobacion.nombre)
        for _clave, _valor in _comprobacion.evidencia.items():
            print("     {0} = {1}".format(_clave, _valor))

### 1.7 Herramienta auxiliar: leer el grafo dibujándolo

El enunciado advierte que *«no se acepta una imagen como evidencia: deben
entregarse estructuras de datos verificables»*. Las once comprobaciones
anteriores son la evidencia; este dibujo **no demuestra nada**. Se incluye por
dos razones distintas y ambas legítimas:

1. **Para leer la representación.** La idea central del modelado es que
   *no hay paredes almacenadas en ninguna parte*. El diccionario guarda
   únicamente pasillos, y **una pared es exactamente la ausencia de la arista
   correspondiente**. El dibujo hace visible esa equivalencia: para cada
   frontera entre dos celdas pregunta si la arista existe, y si no existe traza
   un muro.
2. **Para detectar un error grosero de un vistazo.** Una prueba dice *qué*
   falló; un dibujo sugiere *por qué*. Si la comprobación de conectividad
   fallara, ver una región cerrada ahorra media hora de depuración.

La función recibe las marcas como un **diccionario** `{celda: texto}` y no como
una lista de celdas. Eso no es un capricho: en la sección 7 el algoritmo de Lee
necesitará superponer sobre cada celda su **tiempo de llegada** del frente de
onda, no un simple asterisco. Diseñar ahora la firma general evita reescribir
la función después.

In [ ]:
# --------------------------------------------------------------------------
# Herramienta auxiliar: dibujar el laberinto en texto
#
# El enunciado advierte que "no se acepta una imagen como evidencia": las
# propiedades se demuestran con las comprobaciones de arriba, no con un dibujo.
# Pero un dibujo sirve para LEER el grafo y para detectar a simple vista un
# error grosero, asi que se incluye como apoyo, no como prueba.
#
# La clave para entender la representacion: NO HAY PAREDES ALMACENADAS. El
# diccionario solo guarda pasillos, y una pared es exactamente la AUSENCIA de
# la arista correspondiente. Por eso el dibujo pregunta, para cada frontera
# entre dos celdas, si la arista existe.
#
# `marcas` permite superponer texto sobre las celdas: un asterisco para el
# camino, o numeros para el tiempo de llegada del frente de onda de Lee en la
# seccion 7. Por eso la funcion recibe un diccionario y no una lista.
# --------------------------------------------------------------------------

def dibujar_laberinto(grafo, filas, columnas, marcas=None, ancho=3):
    """Devuelve el laberinto como una cadena de texto con paredes.

    - `marcas`: dict {(fila, columna): texto} que se centra dentro de la celda.
    - `ancho`: caracteres de ancho interior de cada celda. 3 se lee bien en
      laberintos pequenos; 1 hace caber una rejilla de 25 columnas en pantalla.
    """
    marcas = marcas or {}
    barra = "-" * ancho
    hueco = " " * ancho

    lineas = ["+" + (barra + "+") * columnas]
    for fila in range(filas):
        interior = "|"
        inferior = "+"
        for columna in range(columnas):
            celda = (fila, columna)
            texto = str(marcas[celda]).center(ancho)[:ancho] if celda in marcas else hueco
            interior += texto
            # Pared derecha: existe si la arista hacia la celda de al lado NO esta.
            interior += " " if (fila, columna + 1) in grafo[celda] else "|"
            # Pared inferior: idem hacia abajo.
            inferior += (hueco if (fila + 1, columna) in grafo[celda] else barra) + "+"
        lineas.append(interior)
        lineas.append(inferior)
    return "\n".join(lineas)


def marcas_de_camino(camino, simbolo="*"):
    """Convierte una secuencia de celdas en marcas para `dibujar_laberinto`."""
    return {celda: simbolo for celda in camino}


# --- Demostracion 1: un laberinto pequeno, legible celda por celda ---------
_GRAFO_DEMO = LaberintoHuntKill(5, 5, 97950).generar()

print("LABERINTO 5x5 (semilla 97950)")
print(dibujar_laberinto(_GRAFO_DEMO, 5, 5))
print()
print("El generador no devuelve ese dibujo, devuelve este diccionario:")
for _celda in [(0, 0), (2, 2), (4, 4)]:
    print("   grafo[{0}] = {1}".format(_celda, sorted(_GRAFO_DEMO[_celda])))
print()
print("Lectura: desde (0, 0) solo se puede pasar a {0}; hacia las otras dos "
      "celdas vecinas hay pared, es decir, NO hay arista.".format(
          sorted(_GRAFO_DEMO[(0, 0)])))

# --- Demostracion 2: el camino unico, superpuesto sobre el mismo laberinto -
_CAMINO_DEMO = caminos_simples(_GRAFO_DEMO, (0, 0), (4, 4), tope=1)[0]
print()
print("UNICO CAMINO DE (0,0) A (4,4): {0} celdas, {1} pasos".format(
    len(_CAMINO_DEMO), len(_CAMINO_DEMO) - 1))
print(dibujar_laberinto(_GRAFO_DEMO, 5, 5, marcas_de_camino(_CAMINO_DEMO)))
print()
print("Como secuencia de estados, que es lo que deberan devolver los siete")
print("algoritmos de la Version 1:")
print("   " + " -> ".join(str(c) for c in _CAMINO_DEMO))

# --- Demostracion 3: la rejilla del tamano de la instancia individual ------
# Se usa ancho=1 para que 25 columnas quepan en pantalla. La instancia formal,
# con su inicio y su meta, se define en la seccion 2; aqui solo se comprueba
# que el dibujo escala y que el camino unico sigue siendo unico.
_GRAFO_20x25 = LaberintoHuntKill(20, 25, 97950).generar()
_CAMINO_20x25 = caminos_simples(_GRAFO_20x25, (0, 0), (19, 24), tope=1)[0]
print()
print("REJILLA 20x25 (semilla 97950) CON EL CAMINO DE (0,0) A (19,24)")
print("{0} celdas de camino, {1} pasos".format(
    len(_CAMINO_20x25), len(_CAMINO_20x25) - 1))
print(dibujar_laberinto(_GRAFO_20x25, 20, 25,
                        marcas_de_camino(_CAMINO_20x25), ancho=1))

## 2. Instancia individual

Cada estudiante debe construir una instancia reproducible:

- `semilla = últimos 5 dígitos del código estudiantil`;
- `filas = 20 + último dígito`;
- `columnas = 20 + penúltimo dígito`;
- inicio y meta deben estar en cuadrantes opuestos, pero no necesariamente en las esquinas.

Registre esos valores en una tabla. El docente podrá cambiar cualquiera de ellos durante la defensa.

| Parámetro | Valor |
|---|---|
| Semilla | |
| Filas | |
| Columnas | |
| Inicio | |
| Meta | |

In [ ]:
# Construya aquí su instancia individual sin modificar LaberintoHuntKill.

## 3. Formulación formal del problema

Defina rigurosamente:

| Componente | Especificación por completar |
|---|---|
| Conjunto de estados | |
| Estado inicial | |
| Acciones aplicables | |
| Modelo de transición | |
| Prueba de objetivo | |
| Costo de paso | |
| Costo de una solución | |

Además:

1. diferencie **estado**, **nodo de búsqueda** y **celda dibujada**;
2. indique cuándo dos nodos distintos pueden contener el mismo estado;
3. explique por qué debe usarse búsqueda en grafo;
4. establezca invariantes que deben cumplirse durante una búsqueda.

## 4. Contrato común de las tres versiones

Todos los algoritmos deben entregar un registro equivalente a:

```text
ResultadoBusqueda
    encontrado: booleano
    camino: secuencia de estados
    acciones: secuencia de acciones
    costo: número
    profundidad: entero
    expandidos: entero
    generados: entero
    repetidos_descartados: entero
    frontera_maxima: entero
    tiempo_ms: número
```

### Convenciones obligatorias

- Un estado se cuenta como **expandido** al retirarlo de la frontera y generar sus sucesores.
- Un nodo meta retirado de la frontera no se cuenta como expandido si no se generan sucesores.
- **Generado** significa nodo sucesor construido, aunque luego sea descartado.
- La política para estados repetidos debe documentarse.
- El camino debe incluir inicio y meta.
- Si no hay solución, `camino` debe ser vacío y `encontrado` falso.

Estas convenciones son necesarias para comparar implementaciones de manera justa.

## 5. Pseudocódigo base: búsqueda en grafo

El siguiente esquema no especifica la estructura de la frontera ni resuelve el manejo de costos. Debe especializarlo y justificar cada decisión.

```text
BUSQUEDA-GRAFO(problema, frontera):
    nodo_inicial ← crear_nodo(problema.estado_inicial)
    insertar(frente, nodo_inicial)
    alcanzados ← estructura apropiada

    mientras frontera no esté vacía:
        nodo ← extraer_segun_politica(frontera)

        si ES_META(nodo.estado):
            devolver RECONSTRUIR(nodo)

        si nodo debe expandirse:
            para cada acción aplicable:
                hijo ← construir sucesor
                decidir si insertar, reemplazar o descartar hijo

        actualizar métricas

    devolver FRACASO
```

### Preguntas de diseño

1. ¿Cuándo debe marcarse un estado: al generarlo o al expandirlo?
2. ¿La respuesta cambia entre BFS, DFS y UCS?
3. ¿Qué información mínima debe almacenar cada nodo?
4. ¿Cómo se reconstruye el camino sin copiar listas completas en cada nodo?
5. ¿Cómo se resuelve un camino más barato hacia un estado ya descubierto?

# Versión 1 — Implementación desde cero

No puede utilizar bibliotecas de búsqueda o grafos. Se permiten únicamente estructuras estándar como listas, diccionarios, conjuntos, `deque` y colas de prioridad.

Debe implementar:

1. DFS iterativa;
2. BFS;
3. búsqueda de costo uniforme;
4. búsqueda limitada en profundidad;
5. profundización iterativa;
6. búsqueda bidireccional;
7. Lee o expansión por frente de onda.

La versión debe separar, como mínimo:

- representación del problema;
- representación de nodos;
- política de frontera;
- control de repetidos;
- reconstrucción del camino;
- recolección de métricas.

## 6. Pseudocódigo que debe traducirse

### DFS y BFS

```text
inicializar frontera con el nodo inicial
inicializar alcanzados

mientras haya nodos:
    extraer según disciplina LIFO o FIFO
    comprobar objetivo
    expandir
    insertar estados nuevos
```

La diferencia no debe quedar dispersa por todo el programa. Diseñe una abstracción que permita cambiar la política de frontera.

### Costo uniforme

```text
frontera ← cola de prioridad ordenada por g(n)
mejor_costo[inicial] ← 0

mientras frontera no esté vacía:
    extraer nodo con menor g
    ignorar entradas obsoletas
    comprobar objetivo
    para cada sucesor:
        nuevo_costo ← g(nodo) + costo_de_paso
        si mejora el costo conocido:
            actualizar costo, padre y frontera
```

### Profundización iterativa

```text
para límite = 0, 1, 2, ...:
    resultado ← BUSQUEDA-LIMITADA(problema, límite)
    si resultado es solución: devolverlo
    si resultado es fracaso definitivo: terminar
```

Debe distinguir **corte** de **fracaso**.

### Bidireccional

```text
crear una frontera desde inicio y otra desde meta
expandir de forma equilibrada
detectar intersección entre regiones alcanzadas
unir los dos caminos respetando su orientación
```

No se acepta ejecutar dos búsquedas completas y comparar al final.

In [ ]:
# VERSIÓN 1: implemente aquí su solución desde cero.

## 7. Lee: propagación de un frente luminoso

No se modela un único rayo que rebota. Se modela una onda que avanza simultáneamente por todos los corredores accesibles.

```text
LEE(grafo, inicio, meta):
    etiqueta[inicio] ← 0
    frente_actual ← {inicio}

    mientras frente_actual no esté vacío y meta no tenga etiqueta:
        frente_siguiente ← vacío
        para cada celda del frente_actual:
            para cada vecino transitable sin etiqueta:
                etiqueta[vecino] ← etiqueta[celda] + 1
                registrar procedencia o dirección
                agregar vecino al frente_siguiente
        frente_actual ← frente_siguiente

    si meta no tiene etiqueta: devolver fracaso
    reconstruir desde meta siguiendo etiquetas decrecientes
```

### Requisitos adicionales

- Dibuje al menos ocho instantes del frente de onda.
- Use una escala de color que represente el tiempo de llegada.
- Compruebe que la etiqueta de la meta coincide con la profundidad de BFS.
- Explique formalmente por qué Lee es BFS en un grafo no ponderado.
- Analice qué deja de funcionar cuando los costos no son unitarios.

In [13]:
# Implemente y visualice aquí el algoritmo de Lee.

# Versión 2 — SimpleAI

Instalación orientativa:

```text
pip install simpleai
```

Consulte la documentación oficial. Debe crear una subclase de `SearchProblem` y definir, como mínimo:

| Método de SimpleAI | Responsabilidad en el laberinto |
|---|---|
| `actions(state)` | Producir únicamente movimientos legales |
| `result(state, action)` | Calcular el estado sucesor sin mutar el original |
| `is_goal(state)` | Reconocer la celda meta |
| `cost(state, action, state2)` | Devolver el costo del corredor |

Instancie el problema con un estado inicial inmutable y use `graph_search=True`.

Debe ejecutar y estudiar:

- `depth_first`;
- `breadth_first`;
- `uniform_cost`;
- `limited_depth_first`;
- `iterative_limited_depth_first`.

### Exigencia avanzada

SimpleAI devuelve un nodo solución, pero las métricas del contrato no aparecen automáticamente con la misma definición. Investigue su mecanismo de *viewer* o diseñe instrumentación externa **sin modificar el código fuente instalado de la biblioteca**. Documente cualquier diferencia entre las métricas de SimpleAI y las de su versión.

No copie la estructura de su buscador desde cero dentro de esta versión: el propósito es construir correctamente el adaptador del problema y comprender la biblioteca.

In [ ]:
# VERSIÓN 2: modele el problema y ejecute SimpleAI.

# Versión 3 — AIMA-Python

Repositorio de referencia: `aimacode/aima-python`.

Debe modelar el laberinto como una subclase de `Problem`. Investigue y documente los contratos de:

| Método de AIMA-Python | Decisión requerida |
|---|---|
| `actions(state)` | Formato de las acciones y orden determinista |
| `result(state, action)` | Transición entre celdas |
| `goal_test(state)` | Prueba de objetivo |
| `path_cost(c, state1, action, state2)` | Acumulación de costos |

Compare al menos:

- `depth_first_graph_search`;
- `breadth_first_graph_search`;
- `uniform_cost_search`;
- `depth_limited_search`;
- `iterative_deepening_search`;
- `bidirectional_search`.

### Exigencia avanzada

Instrumente los algoritmos sin alterar permanentemente el repositorio. Puede utilizar una subclase del problema, envoltorios o contadores cuidadosamente definidos. Explique por qué contar llamadas a `actions` no siempre coincide con contar nodos generados.

Compare la representación del nodo de AIMA-Python con su propia clase de nodo.

In [ ]:
# VERSIÓN 3: adapte el problema a AIMA-Python.

## 8. Pruebas de aceptación

El estudiante debe implementar pruebas automáticas equivalentes a las siguientes especificaciones, sin recibir su solución:

### Validez de un camino

Para cada resultado exitoso:

1. el primer estado es el inicio;
2. el último estado es la meta;
3. cada pareja consecutiva aparece como arista en el grafo;
4. el costo reportado coincide con la suma de costos;
5. aplicar las acciones reproduce exactamente los estados.

### Concordancia entre versiones

- En costos unitarios: `longitud(BFS) = longitud(UCS) = etiqueta_meta(Lee)`.
- En un árbol: todas las soluciones válidas contienen la misma secuencia de estados.
- Con ciclos: `longitud(BFS) ≤ longitud(DFS)` para una DFS que encuentre solución.
- Con costos positivos: `costo(UCS) ≤ costo(cualquier solución BFS)`.
- Las tres versiones deben concordar en existencia de solución y costo óptimo.

### Casos límite obligatorios

- inicio igual a meta;
- laberinto de `1×1`;
- meta inalcanzable después de eliminar corredores;
- múltiples entradas obsoletas en UCS;
- dos posibles puntos de encuentro bidireccional;
- límite exacto, insuficiente y excesivo en búsqueda limitada.

In [ ]:
# Escriba aquí su batería de pruebas.

## 9. Segunda familia de problemas: ciclos y costos

Hunt-and-Kill produce un árbol. Allí existe un único camino entre dos estados y no se aprecia plenamente la optimalidad. Diseñe, sin modificar el generador original, dos transformaciones:

### A. Laberinto con ciclos

Abra aleatoriamente entre 5% y 12% de las paredes internas que no sean corredores. Debe conservar:

- adyacencias ortogonales;
- simetría del grafo;
- reproducibilidad;
- al menos un ciclo comprobable.

### B. Laberinto ponderado

Asigne costos positivos a corredores o terrenos. Construya deliberadamente un caso donde:

```text
camino con menos pasos ≠ camino de menor costo
```

No use pesos negativos. Justifique si los costos pertenecen a celdas, acciones o aristas y mantenga esa decisión en las tres versiones.

In [ ]:
# Implemente aquí ciclos y costos; no modifique la clase suministrada.

## 10. Diseño experimental

Ejecute como mínimo **30 instancias** por configuración y reporte mediana y rango intercuartílico, no solamente un caso.

| Factor | Niveles mínimos |
|---|---|
| Tamaño | 15×15, 25×25, 40×40 |
| Topología | árbol, 5% ciclos, 10% ciclos |
| Costos | unitarios, ponderados |
| Posición | esquinas, interior, cercanas |
| Algoritmos | todos los exigidos que sean aplicables |

Debe controlar semillas y separar tiempo de generación del tiempo de búsqueda.

### Gráficas obligatorias

1. expandidos frente a número de estados;
2. frontera máxima frente a profundidad de solución;
3. tiempo frente a tamaño;
4. costo y longitud de las soluciones;
5. mapa de calor de Lee;
6. comparación cruzada de las tres versiones.

### Discusión obligatoria

- ¿Qué métrica resulta más estable que el tiempo?
- ¿Cuándo la búsqueda bidireccional pierde su ventaja?
- ¿Por qué IDDFS repite trabajo y aun así puede ser conveniente?
- ¿Qué efecto tiene el orden de sucesores sobre DFS?
- ¿Qué parte de las diferencias se debe al algoritmo y cuál a la biblioteca?

In [14]:
# Construya aquí su protocolo experimental, tablas y gráficas.

## 11. Análisis teórico

Complete y justifique esta tabla usando $b$ como factor de ramificación, $d$ como profundidad de la solución menos profunda y $m$ como profundidad máxima.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones |
|---|---|---|---|---|---|
| DFS | | | | | |
| BFS | | | | | |
| UCS | | | | | |
| DLS | | | | | |
| IDDFS | | | | | |
| Bidireccional | | | | | |
| Lee | | | | | |

Relacione después las cotas teóricas con las mediciones. Una tabla memorizada sin conexión con los experimentos no recibe puntaje completo.

## 12. Preguntas para la defensa

El docente seleccionará algunas al azar:

1. Muestre en memoria la diferencia entre frontera y alcanzados.
2. Cambie BFS a DFS modificando solamente la política apropiada.
3. Explique un caso en el que marcar visitados demasiado tarde duplique trabajo.
4. Explique por qué terminar UCS al generar la meta puede ser incorrecto.
5. Reconstruya un camino usando padres sin almacenar caminos completos.
6. Muestre por qué dos BFS bidireccionales requieren orientar correctamente los padres.
7. Explique por qué Lee no es un rayo geométrico.
8. Adapte el problema a una meta múltiple.
9. Introduzca una celda bloqueada durante la ejecución y analice qué debe recalcularse.
10. Compare `SearchProblem` de SimpleAI con `Problem` de AIMA-Python.
11. Diagnostique una discrepancia de métricas entre dos versiones.
12. Argumente por qué un laberinto perfecto oculta diferencias de calidad de ruta.

## 13. Entrega

El cuaderno final debe contener:

1. auditoría del generador;
2. formulación formal;
3. versión desde cero;
4. versión SimpleAI;
5. versión AIMA-Python;
6. pruebas automáticas;
7. extensión con ciclos y costos;
8. protocolo experimental y gráficas;
9. análisis teórico;
10. conclusiones y referencias;
11. enlace al historial de trabajo o bitácora incorporada.

Todo el cuaderno debe ejecutarse desde el inicio en un entorno limpio. Las celdas fuera de orden, dependencias implícitas y resultados pegados manualmente se consideran defectos reproducibles.

### Rúbrica

| Criterio | Peso |
|---|---:|
| Modelado, invariantes y auditoría de Hunt-and-Kill | 10% |
| Versión desde cero y estructuras de datos | 25% |
| Adaptación correcta a SimpleAI | 12% |
| Adaptación correcta a AIMA-Python | 13% |
| Pruebas, casos límite y concordancia | 15% |
| Experimentos, métricas y visualización | 12% |
| Análisis teórico y conclusiones | 8% |
| Defensa, trazabilidad y calidad del cuaderno | 5% |

Una solución que produzca un camino pero no satisfaga el contrato, las pruebas o la defensa no se considera completa.

## 14. Referencias de partida

- Russell, S. J. y Norvig, P. *Artificial Intelligence: A Modern Approach*, 4.ª edición, capítulo 3.
- SimpleAI, documentación de problemas y búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/docs/search_problems.rst>
- SimpleAI, implementación oficial de búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/simpleai/search/traditional.py>
- AIMA-Python, repositorio oficial: <https://github.com/aimacode/aima-python>
- AIMA-Python, módulo de búsqueda: <https://github.com/aimacode/aima-python/blob/master/aima/search.py>
- C. Y. Lee, “An Algorithm for Path Connections and Its Applications”, 1961.
- A. Adamatzky, “Physical maze solvers. All twelve prototypes implement 1961 Lee algorithm”, 2016: <https://arxiv.org/abs/1601.04672>

Las referencias orientan el estudio de las interfaces; no sustituyen la explicación de las decisiones tomadas.